In [1]:
import json
import re
import shutil
import os
from typing import List
from PIL import Image,ImageDraw
from pydantic import BaseModel
import pdfplumber
from transformers import pipeline 
import torch
import cv2
import numpy as np
import easyocr
from transformers import pipeline
# from pipe_fn import pipe
from output_utils import save_split_output


from confidence_utils import (
    calculate_eob_confidence,
    _unwrap_vlm_output,calculate_model_confidence
)
# =========================================================
# LOAD MODEL
# =========================================================

pipe = pipeline(
    "image-text-to-text",
    model="Qwen/Qwen3-VL-2B-Instruct",
    device_map="auto"
)



import warnings
warnings.simplefilter("ignore")

# =========================
# 1. PYDANTIC SCHEMA
# =========================

class Service(BaseModel):
    service_code: str = ""
    submitted_amount: str = ""
    allowed_amount: str = ""
    paid_amount: str = ""
    patient_owes: str = ""

class Totals(BaseModel):
    total_submitted: str = ""
    total_allowed: str = ""
    total_paid: str = ""
    total_patient_owes: str = ""

class PatientExtraction(BaseModel):
    EOB_ID: str = ""
    patient_name: str = ""
    relationship: str = ""
    dob: str = ""
    date_of_service: str = ""
    services: List[Service] = []
    totals: Totals = Totals()

class EOBExtraction(BaseModel):
    patients: List[PatientExtraction] = []

# =========================
# 1. COLUMN MASKING HELPERS
# =========================
 
def find_negotiated_fee_column_bounds(page, region_top, region_bottom):

    words = page.extract_words()
    region_words = [
        w for w in words
        if float(w["top"]) >= region_top and float(w["bottom"]) <= region_bottom
    ]
 
    # --- Find "Negotiated" header word ---
    neg_header_words = [
        w for w in region_words
        if "negotiated" in w["text"].strip().lower()
    ]
    if not neg_header_words:
        return None
 
    # The topmost occurrence is the column header
    neg_header_words = sorted(neg_header_words, key=lambda w: float(w["top"]))
    neg_word = neg_header_words[0]
    header_top = float(neg_word["top"])
    header_bottom = float(neg_word["bottom"])
 
    # Collect all words on the same header row (within ±6 pts vertically)
    header_row_words = [
        w for w in region_words
        if abs(float(w["top"]) - header_top) < 6
    ]
 
    # Also grab the "Fee" word on the NEXT line if it belongs to this header
    # (some PDFs wrap "Negotiated\nFee" across two lines)
    fee_words_below = [
        w for w in region_words
        if w["text"].strip().lower() == "fee"
        and float(w["top"]) > header_bottom
        and float(w["top"]) < header_bottom + 15  # within ~15 pts
        and abs(float(w["x0"]) - float(neg_word["x0"])) < 30
    ]
 
    col_words = [
        w for w in header_row_words
        if w["text"].strip().lower() in ("negotiated", "fee", "negotiatedfee")
    ] + fee_words_below
 
    col_x0 = min(float(w["x0"]) for w in col_words) - 4  # small left pad
    col_x1_raw = max(float(w["x1"]) for w in col_words)
 
    # Determine right boundary: midpoint to next column header
    other_header_words = [
        w for w in header_row_words
        if w not in col_words and float(w["x0"]) > col_x1_raw
    ]
    if other_header_words:
        next_col_x0 = min(float(w["x0"]) for w in other_header_words)
        col_x1 = (col_x1_raw + next_col_x0) / 2
    else:
        col_x1 = col_x1_raw + 20
 
    # --- Y bounds: start just below the header row ---
    # header_bottom is the bottom of "Negotiated"; use the lower of the two
    # header lines if "Fee" wrapped to a second line
    if fee_words_below:
        data_y0 = max(float(w["bottom"]) for w in fee_words_below)
    else:
        data_y0 = header_bottom
 
    # --- Y bounds: end at bottom of "Totals" row, or region_bottom ---
    totals_words = [
        w for w in region_words
        if w["text"].strip().lower() == "totals"
        and float(w["top"]) > data_y0
    ]
    if totals_words:
        # Use the bottom of the last Totals row
        data_y1 = max(float(w["bottom"]) for w in totals_words)
    else:
        data_y1 = region_bottom
 
    return {
        "pdf_x0": col_x0,
        "pdf_x1": col_x1,
        "pdf_y0": data_y0,   # PDF absolute coords
        "pdf_y1": data_y1,
    }
 
COLUMN_HEADERS = [
    "date of",
    "service code",
    "you submitted",
    "negotiated",
    "allowed",
    "metlife",
    "patient owes"
]

def find_column_boundaries(page, region_top, region_bottom):
    words = page.extract_words()

    region_words = [
        w for w in words
        if float(w["top"]) >= region_top and float(w["bottom"]) <= region_bottom
    ]

    column_positions = []

    for header in COLUMN_HEADERS:
        for w in region_words:
            if header in w["text"].lower():
                column_positions.append(float(w["x0"]))
                break  # take first occurrence only

    # Sort left → right
    column_positions = sorted(column_positions)

    return column_positions

def draw_column_lines(image, col_x_positions, page_width, pdf_region_top, pdf_region_bottom):
    img_w, img_h = image.size
    region_h_pdf = pdf_region_bottom - pdf_region_top

    scale_x = img_w / page_width

    draw = ImageDraw.Draw(image)

    for x in col_x_positions:
        pix_x = int(x * scale_x)

        draw.line(
            [(pix_x, 0), (pix_x, img_h)],
            fill="#505050",   # dark gray
            width=6           # 🔥 thick line
        )

    return image
 
def mask_negotiated_column(image, bounds, page_pdf_width,
                            pdf_region_top, pdf_region_bottom):
    """
    Draw a white rectangle over the Negotiated Fee column data rows only.
 
    bounds: dict from find_negotiated_fee_column_bounds()
    The cropped image maps:
      x: [0, page_pdf_width]  → [0, img_w]
      y: [pdf_region_top, pdf_region_bottom] → [0, img_h]
    """
    img_w, img_h = image.size
    region_h_pdf = pdf_region_bottom - pdf_region_top
 
    scale_x = img_w / page_pdf_width
    scale_y = img_h / region_h_pdf
 
    # X coords
    pix_x0 = int(bounds["pdf_x0"] * scale_x)
    pix_x1 = int(bounds["pdf_x1"] * scale_x)
 
    # Y coords — shift by region_top so they're relative to the crop
    pix_y0 = int((bounds["pdf_y0"] - pdf_region_top) * scale_y)
    pix_y1 = int((bounds["pdf_y1"] - pdf_region_top) * scale_y)
 
    # Clamp to image bounds
    pix_x0 = max(0, pix_x0)
    pix_x1 = min(img_w, pix_x1)
    pix_y0 = max(0, pix_y0)
    pix_y1 = min(img_h, pix_y1)
 
    # draw = ImageDraw.Draw(image)
    # draw.rectangle([pix_x0, pix_y0, pix_x1, pix_y1], fill="white")

    draw = ImageDraw.Draw(image)
    # Light gray instead of pure white
    draw.rectangle([pix_x0, pix_y0, pix_x1, pix_y1], fill="#F0F0F0")
    
    # Add subtle vertical separator lines
    draw.line([pix_x0-2, pix_y0, pix_x0-2, pix_y1], fill="#D0D0D0", width=3)
    draw.line([pix_x1+2, pix_y0, pix_x1+2, pix_y1], fill="#D0D0D0", width=3)
    
    return image

    # return image
 
 
# =========================
# 2. EXISTING HELPERS
# =========================
 
def find_keyword_positions(lines, keyword):
    keyword = keyword.lower()
    return sorted([
        ln["top"] for ln in lines
        if ln["text"].strip().lower().startswith(keyword)
    ])
 
 
def clean_claim_id(text):
    match = re.search(r"\d[\d\s]+\d", text)
    return match.group(0).strip() if match else "unknown"
 
 
def extract_claim_id(page, top, bottom):
    region = page.within_bbox((0, top, page.width, bottom))
    text = region.extract_text() or ""
    match = re.search(r"Claim:\s*(.+)", text)
    return clean_claim_id(match.group(1)) if match else "unknown"

# def extract_patient_name(page, top, bottom):

#     region = page.within_bbox((0, top, page.width, bottom))
#     text = region.extract_text() or ""

#     match = re.search(
#         r"Name\s*/\s*Relationship:\s*(.*?)\s*/",
#         text,
#         flags=re.IGNORECASE | re.DOTALL
#     )

#     if match:
#         return " ".join(match.group(1).split())

#     return ""


def extract_patient_name(page, top, bottom):

    region = page.within_bbox((0, top, page.width, bottom))
    text = region.extract_text() or ""

    match = re.search(
        r"Name\s*/\s*Relationship:\s*(.*?)\s*/",
        text,
        flags=re.IGNORECASE | re.DOTALL
    )

    if match:
        patient_name = " ".join(match.group(1).split())

        # Remove middle initials like "A.", "B.", etc.
        patient_name = re.sub(r"\b[A-Z]\.\s*", "", patient_name)

        return patient_name.strip()

    return ""
 
 
def crop_region_to_image(page, top, bottom):
    cropped = page.within_bbox((0, top, page.width, bottom))
    im = cropped.to_image(resolution=300)
    return im.original
 
 
def merge_images_vertically(images):
    max_width = max(img.width for img in images)
    total_height = sum(img.height for img in images)
    merged = Image.new("RGB", (max_width, total_height), (255, 255, 255))
    y_offset = 0
    for img in images:
        merged.paste(img, (0, y_offset))
        y_offset += img.height
    return merged
 
 
 
def process_pdf(pdf_path, mask_negotiated=True):
    claims = {}
    patient_names = {}
 
    with pdfplumber.open(pdf_path) as pdf:
        for page_idx, page in enumerate(pdf.pages):
            lines = page.extract_text_lines()
            page_height = page.height
            page_width = page.width
 
            claim_tops = find_keyword_positions(lines, "Name/Relationship")
            note_tops = find_keyword_positions(lines, "additional note")
 
            if not claim_tops:
                continue
 
            for i, claim_top in enumerate(claim_tops):
                crop_bottom = page_height
 
                if i + 1 < len(claim_tops):
                    crop_bottom = claim_tops[i + 1]
 
                for note_top in note_tops:
                    if claim_top < note_top < crop_bottom:
                        crop_bottom = note_top
                        break
                            
                claim_id = extract_claim_id(page, claim_top, crop_bottom)
                patient_name = extract_patient_name(
                    page,
                    claim_top,
                    crop_bottom
                )

                patient_names[claim_id] = patient_name

                if not claim_id:
                    claim_id = f"page{page_idx}"
 
                # Crop region to PIL image
                img = crop_region_to_image(page, claim_top, crop_bottom)

              
 
                # --- MASKING: find and blank the Negotiated Fee column ---
                if mask_negotiated:
                    col_bounds = find_negotiated_fee_column_bounds(
                        page, claim_top, crop_bottom
                    )
                    if col_bounds:
                        img = mask_negotiated_column(
                            img,
                            col_bounds,
                            page_width,
                            claim_top, crop_bottom
                        )

                metlife_word, patient_word = find_header_positions(
                    page,
                    claim_top,
                    crop_bottom
                )

                draw = ImageDraw.Draw(img)

                img_w, img_h = img.size

                if metlife_word and patient_word:

                    metlife_right = pdf_to_image_x(
                        float(metlife_word["x1"]),
                        page.width,
                        img_w
                    )

                    patient_left = pdf_to_image_x(
                        float(patient_word["x0"]),
                        page.width,
                        img_w
                    )

                    text = "deductible"

                    # Center the text between the two columns
                    text_width = draw.textlength(text)

                    text_x = int((metlife_right + patient_left) / 2 - text_width / 2)

                    text_y = int(
                        (float(metlife_word["bottom"]) - claim_top)
                        * img_h
                        / (crop_bottom - claim_top)
                    ) + 5

                    font = ImageFont.truetype("arial.ttf", 36)   # 24 = font size

                    draw.text(
                        (text_x, text_y),
                        text,
                        fill="black",
                        font=font
                    )


                    img = draw_deductible_line(
                            img,
                            page,
                            claim_top,
                            crop_bottom,
                            metlife_word,
                            patient_word
                        )


                    # ✅ ADD THIS
                # col_positions = find_column_boundaries(page, claim_top, crop_bottom)
                # if col_positions:
                #     img = draw_column_lines(
                #         img,
                #         col_positions,
                #         page_width,
                #         claim_top,
                #         crop_bottom
                #     )
 
                if claim_id not in claims:
                     claims[claim_id] = []
                claims[claim_id].append(img)
 
    return claims, patient_names
 
def save_merged_images(claims, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    paths = []
    for claim_id, images in claims.items():
        merged = merge_images_vertically(images)
        safe_name = re.sub(r"[^\w\-]", "_", claim_id)
        output_path = os.path.join(output_dir, f"{safe_name}.png")
        merged.save(output_path)
        print(f"  Saved image: {output_path}")
        paths.append(output_path)
    return paths

#draw column lines
def make_table(image_path):
    image_bgr = cv2.imread(image_path)

    # Convert BGR to RGB
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)    
    img = cv2.imread(image_path, 0)
    # print(f"{image_path = }")
    _, thresh = cv2.threshold(img, 150, 255, cv2.THRESH_BINARY_INV)
    # print(f"{thresh.shape}")
    sums = np.sum(thresh, axis=1)
    thres_line = (thresh.shape[1]*255)*0.6
    lines = np.where(sums>thres_line)
    thresh_h_line = thresh.shape[0]*0.05
    thresh_col_lines = thresh.shape[1]*0.05

    lines_h = list(lines[0])
    lines_h = [int(i) for i in lines_h]

    targate_words=["Service code","submitted","Negotiated","Allowed","Metlife","Patient"]
    stop_words_hor = ["Continued"]
    column_positions=[]
    row_positions = []

    # NEW: track right-edge of "Metlife" and left-edge of "Patient" separately
    metlife_x1_vals = []
    patient_x0_vals = []

    reader = easyocr.Reader(['en'])
    results = reader.readtext(image_path)

    for detection in results:
        bbox = detection[0]   # [tl, tr, br, bl]
        text = detection[1]

        for words in targate_words:
            if words.lower() in text.lower():
                column_positions.append(bbox[0][0])
                row_positions.append(bbox[0][1])

        # NEW: capture the specific edges we need
        if "metlife" in text.lower():
            metlife_x1_vals.append(max(bbox[1][0], bbox[2][0]))   # right edge
        if "patient" in text.lower():
            patient_x0_vals.append(min(bbox[0][0], bbox[3][0]))   # left edge

        for words in stop_words_hor:
            if words.lower() in text.lower():
                lines_h.insert(0, int(bbox[0][1]))

    # print(f"{lines_h = }")
    lines_h = sorted(lines_h)
    for i_line in range(len(lines_h)): 
        # print(f"{lines_h = }")
        # print(f"{len(lines_h) = }")
        # print(f"{i_line = }")
        if i_line<len(lines_h)-1:
            line = lines_h[i_line]
            if abs(lines_h[i_line] - lines_h[i_line+1])>=thresh_h_line:
                cv2.line(image_bgr, (0, line-2), (thresh.shape[1], line), (0, 0, 0), thickness=1,)
    cv2.line(image_bgr, (0, lines_h[-1]), (thresh.shape[1], lines_h[-1]), (0, 0, 0), thickness=1,)
    

    # print(f"{row_positions = }")
    first_line=min(row_positions)
    # first_line=min(list(lines_h))
     
    last_line=max(list(lines_h))
    
    # for line in column_positions: 
    #     cv2.line(image_bgr, (line-2, first_line), (line,last_line), (0, 0, 0), thickness=1,)
    column_positions = sorted(column_positions)
    # print(f"{thresh_h_line= }")
    for i_line in range(len(column_positions)): 
        # print(f"{lines_h = }")
        # print(f"{len(lines_h) = }")
        # print(f"{i_line = }")
        if i_line<len(column_positions)-1:
            line = column_positions[i_line]
            if abs(column_positions[i_line] - column_positions[i_line+1])>=thresh_col_lines:
                # cv2.line(image_bgr, (0, line-2), (thresh.shape[1], line), (0, 0, 0), thickness=1,)
                cv2.line(image_bgr, (column_positions[i_line], first_line), (column_positions[i_line],last_line), (0, 0, 0), thickness=1,)
            # else: 
            #     print(f"-------------Else statement is runingin the colum filtering")
    cv2.line(image_bgr, (column_positions[-1], first_line), (column_positions[-1],last_line), (0, 0, 0), thickness=1,)
    

    # plt.imshow(image_bgr, cmap = "gray")

    SPACE = 15

    metlife_line_x = None
    patient_line_x = None

    if metlife_x1_vals:
        metlife_line_x = int(max(metlife_x1_vals)) + SPACE
        cv2.line(
            image_bgr,
            (metlife_line_x, first_line),
            (metlife_line_x, last_line),
            (0, 0, 0),
            thickness=1,
        )

    if patient_x0_vals:
        patient_line_x = int(min(patient_x0_vals)) - SPACE
        cv2.line(
            image_bgr,
            (patient_line_x, first_line),
            (patient_line_x, last_line),
            (0, 0, 0),
            thickness=1,
        )
    return image_bgr


def find_header_positions(page, region_top, region_bottom):

    words = page.extract_words()

    region_words = [
        w for w in words
        if region_top <= float(w["top"]) <= region_bottom
    ]

    metlife = None
    patient = None

    for w in region_words:

        txt = w["text"].strip().lower()

        if metlife is None and "metlife" in txt:
            metlife = w

        if patient is None and (
            txt == "patient"
            or "patient owes" in txt
            or "patientowes" in txt
        ):
            patient = w

    return metlife, patient

def pdf_to_image_x(pdf_x, page_width, image_width):
    return int(pdf_x * image_width / page_width)

def draw_deductible_line(
    image,
    page,
    region_top,
    region_bottom,
    metlife_word,
    patient_word,
):
    draw = ImageDraw.Draw(image)

    img_w, img_h = image.size

    metlife_right = pdf_to_image_x(
        float(metlife_word["x1"]),
        page.width,
        img_w
    )

    patient_left = pdf_to_image_x(
        float(patient_word["x0"]),
        page.width,
        img_w
    )

    # halfway between columns
    PADDING = 12

    line_x = metlife_right + PADDING

    draw.line(
        [(line_x, 0), (line_x, img_h)],
        fill="black",
        width=2
    )

    return image



def build_prompt(pdf_name):
    schema_json = json.dumps(PatientExtraction.model_json_schema(), indent=2)

    schema_instruction = f"""
You MUST return output strictly matching this JSON schema:

{schema_json}

RULES:
Output ONLY JSON
No markdown
No explanation
No extra keys
No missing keys
"""

    task_prompt = f"""ROLE: 
    You are a precise OCR extractor who extracts the necessary information in the 
    given image and returns the desired output as json. 

    
CRITICAL RULE — EMPTY CELLS:

If a cell is empty, blank, or has NO clear numeric value:
→ output exactly ""

DO NOT:
- copy from above row
- copy from below row
- copy from totals row
- infer or guess

Even if nearby values exist → DO NOT use them


TASK: 
Your goal is to extract the details of claims in the given Explanation Of Benefits (EOB) image. 

ABSOLUTE RULES — YOU MUST OBEY THESE FOR EVERY IMAGE, NO EXCEPTIONS:
- The image may contain MULTIPLE separate table sections for the SAME patient (repeated "Name/Relationship", "Dentist", headers, and "Continued on next page" notes). 
- You MUST scan the ENTIRE image top-to-bottom and extract EVERY service row from ALL table sections.
- Count every visible horizontal service line that contains a "Date of service" + service code. 
  → If the image shows 4 service lines, you MUST output EXACTLY 4 objects in the "services" array.
  → If the image shows 5 service lines, output exactly 5, etc.
- Duplicates are intentional and mandatory. Even if two or more rows have 100% identical service_code + amounts + description, extract EACH one as a separate object. 
  NEVER skip, merge, deduplicate, or ignore any row because it looks the same as the previous row.
- Never stop extracting when you see repeated headers or "Continued on next page". Continue until you reach the final "Totals" row.

COLUMN RULES (strict):
- Columns are fixed by headers: "You submitted" | "Negotiated" | "Allowed amount" | "MetLife Paid" | "Patient Owes"
- Extract ONLY the value that sits DIRECTLY under each header in that exact row.
- If a monetary cell is blank, empty, or not a clean dollar amount → output exactly "".
- Never shift or copy values left-to-right or from Totals into any service row.




DETAILS TO EXTRACT: 

PATIENT NAME EXTRACTION (STRICT)

Locate the text immediately after "Name/Relationship:".

The format is always:

<Name>/<Relationship>

Extract ONLY the patient's name.

RULES:
- Stop extracting at the FIRST "/" character.
- NEVER include the "/" character.
- NEVER include the relationship after the slash.
- NEVER output words such as:
  Dependent
  Self

- NEVER guess or infer.
- Preserve the name exactly as printed (spacing, punctuation, initials).
- Trim leading/trailing spaces.

Examples:

Name/Relationship: JOELLE CRANE/Dependent
→ patient_name = "JOELLE CRANE"

Name/Relationship: JOHN A. SMITH/Self
→ patient_name = "JOHN A. SMITH"


The output MUST NEVER contain '/' or any relationship text.
If the extracted patient_name contains '/', it is INCORRECT. Re-extract until patient_name contains ONLY the name.


Per patient: 
- EOB_ID -> {pdf_name} 
- patient_name -> The exact "Name" string from "Name/Relationship:" (only the name part before the slash)
- relationship -> The exact "Relationship" string from "Name/Relationship:"
- dob -> "" (always empty)
- date_of_service -> The date from the very first service row on the image
- services: one object per visible service row (extract ALL rows from ALL tables on the image)
- totals: EXACTLY the values shown in the bottom "Totals" row at the very end of the image

Per Service row:
- service_code -> ONLY the first 5 characters (e.g. "D2392"). Discard everything after.
- submitted_amount -> Exact value DIRECTLY UNDER "You submitted" in THIS ROW ONLY. Blank → "". If any word appeard inside this cell instead of "$" amount Only -> ""
- allowed_amount   -> Exact value DIRECTLY UNDER "Allowed amount" in THIS ROW ONLY. Blank → "". If any word appeard inside this cell instead of "$" amount Only -> ""
- paid_amount      -> Exact value DIRECTLY UNDER "MetLife Paid" in THIS ROW ONLY. Blank → "". If any word appeard inside this cell instead of "$" amount Only -> ""
- deductible        -> Exact value DIRECTLY UNDER "deductible" in THIS ROW ONLY. Blank → "", extract what it appears.
- patient_owes     -> Exact value DIRECTLY UNDER "Patient Owes" in THIS ROW ONLY. Blank → "". If any word appeard inside this cell instead of "$" amount Only -> ""

Per Totals (only the very last "Totals" row at the bottom of the image):
- total_submitted     → Exact value under "You submitted"
- total_allowed       → Exact value under "Allowed amount"
- total_paid          → Exact value under "MetLife Paid"
- total_patient_owes  → Exact value under "Patient Owes"
OUTPUT JSON SCHEMA (exactly this structure):

{{

  "patient_name": {{
    "value": "",
    "confidence": 0.0
  }},

  "relationship": {{
    "value": "",
    "confidence": 0.0
  }},

  "dob": {{
    "value": "",
    "confidence": 0.0
  }},

  "date_of_service": {{
    "value": "",
    "confidence": 0.0
  }},

  "services": [

    {{

      "service_code": {{
        "value": "",
        "confidence": 0.0
      }},

      "submitted_amount": {{
        "value": "",
        "confidence": 0.0
      }},

      "allowed_amount": {{
        "value": "",
        "confidence": 0.0
      }},

      "paid_amount": {{
        "value": "",
        "confidence": 0.0
      }},

      "deductible": {{
        "value": "",
        "confidence": 0.0
      }},

      "patient_owes": {{
        "value": "",
        "confidence": 0.0
      }}

    }}

  ],

  "totals": {{

    "total_submitted": {{
      "value": "",
      "confidence": 0.0
    }},

    "total_allowed": {{
      "value": "",
      "confidence": 0.0
    }},

    "total_paid": {{
      "value": "",
      "confidence": 0.0
    }},

    "total_patient_owes": {{
      "value": "",
      "confidence": 0.0
    }}

  }}

}}
 For every extracted field, return:
   - value
   - confidence
 
VALUE + CONFIDENCE RULES:
 
For every field return:
{{
  "value": "",
  "confidence": ""
}}
 
VALUE:
- "value" = the exact text/value visibly present in the specified location.
- Read ONLY from the exact cell/row/column requested.
- Copy exactly as printed; preserve "$" and formatting when visible.
- Never guess, infer, calculate, copy, shift, or use values from another row,
  column, table section, or Totals row.
- If the exact location is blank, missing, or has no clearly readable value:
  value = ""
 
CONFIDENCE:
- "confidence" = confidence that the extracted value is actually present
  in that exact location.
- Use a number from 0.0 to 1.0 based ONLY on visual evidence.
- 1.0 = clearly visible and certain.
- 0.8–0.99 = clearly visible with minor uncertainty.
- 0.5–0.79 = visible but difficult/ambiguous.
- 0.1–0.49 = very unclear.
- 0.0 = blank, missing, or no reliable visual evidence.
 
IMPORTANT:
Confidence is NOT confidence that the value is mathematically correct
or logically expected. It is ONLY confidence that the value shown in
"value" is what is visibly printed in the exact requested location.
 
If value = "":
confidence MUST = 0.0.

"""

    return task_prompt #+ "\n\n" + schema_instruction



def build_feedback_prompt(pdf_name, old_validation, old_output, errors):

    focus_fields = list(set([e["field"] for e in errors]))
    print(f"Focus Fields = {focus_fields}")

    second_chance =  f"""
CRITICAL RULE — EMPTY CELLS:

If a cell is empty, blank, or has NO clear numeric value:
→ output exactly ""
If a cell is empty, blank, or has NO clear numeric value like $0.00 or $51.00:
→ output exactly ""

DO NOT:
- copy from above row
- copy from below row
- copy from totals row
- infer or guess

Even if nearby values exist → DO NOT use them

ROLE:
You made mistakes in extraction.

-----------------------------------
PREVIOUS OUTPUT (DONT COPY VALUES FROM THIS OLD_OUTPUT ONLY USE THIS AS REFERENCE):
{old_output}

VALIDATION ERRORS:
{old_validation}

🚨 MUST FOLLOW:

- IGNORE previous output completely
- DO NOT copy any values from it
- Re-extract values from IMAGE only
-----------------------------------
TARGETED CORRECTION:

Incorrect columns:
{focus_fields}

-----------------------------------
STRICT RULES:

- Each row is independent
- No copying between rows
- No guessing
- No reuse of old output

-----------------------------------


{{
"EOB_ID": "{pdf_name}",
"patient_name": "",
"relationship": "",
"dob": "",
"date_of_service": "",
"services": [
{{
"service_code": "",
"submitted_amount": "",
"allowed_amount": "",
"paid_amount": "",
"patient_owes": ""
}}
],
"totals": {{
"total_submitted": "",
"total_allowed": "",
"total_paid": "",
"total_patient_owes": ""
}}
}}

OUTPUT JSON ONLY
"""

    return second_chance #+ "\n\n" + schema_instruction




# =========================
# 5. JSON CLEANER
# =========================

def extract_json(text):
    text = re.sub(r"json|", "", text)
    text = re.sub(r"<.*?>", "", text)
    start = text.find("{")
    end = text.rfind("}") + 1
    return json.loads(text[start:end])


# =========================
# 6. VALIDATION HELPERS
# =========================

def parse_amount(value: str) -> float:
    """
    Accepts:
    $89.00
    89.00
    $1,234.56
    1,234.56

    Rejects:
    SEE NOTE 1
    Applied to deductible
    ""
    """

    if not value or not isinstance(value, str):
        return 0.0

    value = value.strip()

    # Allow optional $
    pattern = r"^\$?\d{1,3}(,\d{3})*(\.\d{2})?$|^\$?\d+(\.\d{2})?$"

    if not re.fullmatch(pattern, value):
        return 0.0

    try:
        return float(value.replace("$", "").replace(",", ""))
    except:
        return 0.0
    

def validate_patient_totals(patient: dict, patient_name: str):

    services = patient.get("services", [])
    totals   = patient.get("totals", {})

    if not services:
        return False, "", []

    computed_totals = {
        "total_submitted": round(sum(parse_amount(s.get("submitted_amount", "")) for s in services), 2),
        "total_allowed": round(sum(parse_amount(s.get("allowed_amount", "")) for s in services), 2),
        "total_paid": round(sum(parse_amount(s.get("paid_amount", "")) for s in services), 2),
        "total_patient_owes": round(sum(parse_amount(s.get("patient_owes", "")) for s in services), 2),
    }

    field_map = {
        "total_submitted": "submitted_amount",
        "total_allowed": "allowed_amount",
        "total_paid": "paid_amount",
        "total_patient_owes": "patient_owes"
    }

    result_validation = ""
    errors = []
    has_error = False

    print(f"\n🔍 Validation for [{patient_name}]")
    print("-" * 75)

    for total_key, computed_value in computed_totals.items():

        extracted_value = round(parse_amount(totals.get(total_key, "")), 2)

        if computed_value == extracted_value:
            icon = "✅"
            status = "match"
        else:
            icon = "❌"
            status = "MISMATCH"
            has_error = True

            # 🔥 COLUMN LEVEL ERROR
            service_field = field_map[total_key]

            # 🔥 ROW LEVEL DETECTION (heuristic)
            for i, s in enumerate(services):
                val = parse_amount(s.get(service_field, ""))
   
                if computed_value != extracted_value:
                    errors.append({
                        "field": service_field ,
                        "computed": computed_value,
                        "extracted": extracted_value                      
                    })

        line = f"{icon} {total_key:25s} computed={computed_value:<10} | extracted={extracted_value:<10} {status}"
        print(line)
        result_validation += "\n" + line

    print("-" * 75)

    # return (not has_error), result_validation, errors

    if has_error:
        print(f"  ❌ [{patient_name}] Validation FAILED\n")
        return False, result_validation, errors   # 🔥 UPDATED
    else:
        print(f"  ✅ [{patient_name}] Validation PASSED\n")
        return True, result_validation, []        # 🔥 UPDATED
# =========================
# 7. MAIN FUNCTION
# =========================
# -----------------------------------------------------
# DENIAL DETECTION
# -----------------------------------------------------
def check_claim_denied(pdf_path):

    """
    Logic:
    Search ONLY before:
    'Attention Non-contracted Medicare Providers'

    If denied / denial keywords exist before that section,
    return True else False
    """

    denial_keywords=[
        "denied",
        "denial"
    ]
    stop_word="Additional Note(s):"

    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            full_text=page.extract_text()
            if not full_text:
                continue
            searchable_text =full_text.lower()

            # remove notes section
            for word in stop_word:
                if word in searchable_text:
                    searchable_text=searchable_text.split(word)[0]
                    break
            # search keywords
            for keyword in denial_keywords:
                if keyword in searchable_text:
                    print(f"claim denied keyword found:{keyword}")
                    return "denied"
        return "not denied"
    
from PIL import Image, ImageDraw, ImageFont
import re

def clean_money_value(value):
    if not value or not isinstance(value, str):
        return ""

    value = value.strip()

    # Ignore SEE NOTE
    if value.upper().startswith("SEE NOTE"):
        return ""

    # Ignore dates like 03/03/26 or 03-03-2026
    if re.fullmatch(r"\d{1,2}[/-]\d{1,2}[/-]\d{2,4}", value):
        return ""

    # Extract only money values
    match = re.search(r"\$?\s*([\d,]+\.\d{2})", value)

    if match:
        return match.group(1).replace(",", "")

    return ""


def clean_extracted_json(patient_data):
    """
    Cleans all monetary fields in extracted JSON.
    """

    # Service-level fields
    for service in patient_data.get("services", []):
        for field in [
            "submitted_amount",
            "allowed_amount",
            "paid_amount",
            "deductible",
              # keep both spellings if model returns either
            "patient_owes",
        ]:
            if field in service:
                service[field] = clean_money_value(service[field])

    # Totals
    totals = patient_data.get("totals", {})
    for field in [
        "total_submitted",
        "total_allowed",
        "total_paid",
        "total_patient_owes",
    ]:
        if field in totals:
            totals[field] = clean_money_value(totals[field])

    return patient_data


def run_pipeline(pdf_path,output_dir="EOB_OUTPUT/Metlife",error_dir="error_logs/Metlife", max_retries=1, company_name= "Metlife"):
    pdf_file = os.path.basename(pdf_path)
    pdf_name = pdf_file.split("_")[-1].split(".")[0]

    print(f"{'='*60}")
    print(f"Processing PDF: {pdf_file}  (ID: {pdf_name})")
    print(f"{'='*60}")
    
    is_denied = check_claim_denied(pdf_path)
    print(f"claim status : {is_denied}")

    base_output_dir = os.path.join(output_dir, pdf_name)
    os.makedirs(base_output_dir, exist_ok=True)

    error_dir = os.path.join(error_dir)
    os.makedirs(error_dir, exist_ok=True)

    # 🔁 Retry loop (Initial + Feedback pass)
    old_output = None
    old_validation = None

    for attempt in range(max_retries + 1):

        is_feed_back = attempt > 0

        print(f"\n{'-'*50}")
        print(f"ATTEMPT {attempt+1} | Feedback: {is_feed_back}")
        print(f"{'-'*50}")

        # 🔹 Prompt selection
        if is_feed_back:
            print("\n🟡 FEEDBACK MODE ACTIVATED")

            print("\n📤 Previous Output Sent To Model:\n")
            print(old_output[:1000])   # limit to avoid huge logs

            print("\n📊 Validation Errors Sent To Model:\n")
            print(old_validation)

            final_prompt = build_feedback_prompt(pdf_name, old_validation, old_output, all_errors)

            print("\n🧠 FINAL FEEDBACK PROMPT (IMPORTANT):\n")
            print(final_prompt)  # truncate if too long
        else:
            print("Building prompt!")
            final_prompt = build_prompt(pdf_name)

        # 🔹 Step 1: Crop images
        claims, patient_names = process_pdf(pdf_path)

        for claim_id, patient_name in patient_names.items():
            print(f"Claim ID : {claim_id}")
            print(f"Patient  : {patient_name}")
            print("-" * 60)
        pdf_path_save=os.path.join(base_output_dir, "clean_imgs")
        os.makedirs(pdf_path_save, exist_ok=True)
        count=0
        for k,v in claims.items():

            for claim in v:
                claim.save(os.path.join(pdf_path_save,f"{count}.jpg"))
                count+=1
        images_path = save_merged_images(claims, base_output_dir)
        claim_ids = list(claims.keys())

        # 🔹 Step 2: Run model
        all_patients = []
        imgs_paths = os.listdir(pdf_path_save)
        imgs_paths = [os.path.join(pdf_path_save, i) for i in imgs_paths]
        finalImage_path_save=os.path.join(base_output_dir, "imagesToModel")
        os.makedirs(finalImage_path_save, exist_ok=True)

        for idx, img_path in enumerate(images_path):
            image = make_table(img_path)
            # plt.imshow(image)
            # plt.show()
            image = Image.fromarray(image).convert("RGB")
            image.save(os.path.join(finalImage_path_save, f"{idx}.jpg"))
            # image = Image.open(img_path).convert("RGB")
            print(f"  Processing image {idx + 1}: {img_path}")

            messages = [
                {
                    "role": "user",
                    "content": [
                        {"type": "image", "image": image},
                        {"type": "text", "text": final_prompt}
                    ]
                }
            ]
            with torch.no_grad():
                output = pipe(messages, max_new_tokens=1500, temperature=0.0, do_sample=False)

            raw_text = output[0]["generated_text"][1]["content"]

            try:
                patient_data = extract_json(raw_text)

                model_confidence = calculate_model_confidence(patient_data)
                patient_data = _unwrap_vlm_output(patient_data)
                patient_data["_model_confidence"] = model_confidence

                claim_id = claim_ids[idx]

                patient_data["patient_name"] = patient_names.get(claim_id, "")


                # Clean extracted money fields
                patient_data = clean_extracted_json(patient_data)

                if "relationship" not in patient_data:
                    patient_data["relationship"] = ""

                all_patients.append(patient_data)
                print(f"  → Extracted: {patient_data.get('patient_name', 'Unknown')}")

            except Exception as e:
                print(f"  → Failed JSON parse: {e}")
                print(f"     Raw: {raw_text[:300]}")

        # 🔹 Step 3: Build output
        # final_output = [
        #     {
        #     "eob_id": pdf_name,
        #     "payor" : "Metlife",
        #     "claim_status": is_denied,
        #     "patients": all_patients
        #     }
        # ]

        

        # 🔹 Step 3: Build patient-level validation & output
        print(f"\n  Running validation...")

        all_valid = True
        res_val = ""
        all_errors = []

        for patient in all_patients:
            patient_name = patient.get("patient_name", "Unknown")

            is_valid, res_val_, errors = validate_patient_totals(patient, patient_name)

            # ✅ attach validation directly onto the patient dict
            patient["validation"] = {
                "status": is_valid,
                "errors": errors
            }

            all_errors.extend(errors)
            res_val += f"\n{'_'*75}\n" + res_val_

            if not is_valid:
                all_valid = False

        confidence_score = calculate_eob_confidence(all_patients)   # ADD — before popping temp keys

        for patient in all_patients:
            patient.pop("_model_confidence", None)   # ADD

        final_output = [
            {
                "eob_id": pdf_name,
                "payor": "Metlife",
                "claim_status": is_denied,
                "confidence_score": confidence_score,   # CHANGED — was hardcoded "98"
                "pdf_file": pdf_file,
                "patients": all_patients
            }
        ]

        # ✅ If everything passed OR we're on the last allowed attempt → save & return
        if all_valid or attempt == max_retries:

            success_path, failed_path = save_split_output(
                final_output,
                company_name=company_name,
                
                pdf_name=pdf_name,
                pdf_path=pdf_path,
                cropped_dir=base_output_dir,
            )

            print(f"\n📁 Cropped images : {base_output_dir}")
            print(f"✅ Success json   : {success_path}")
            print(f"⚠  Failed json    : {failed_path}")

            if all_valid:
                print(f"\n✅ SUCCESS on attempt {attempt+1}")
            else:
                print(f"\n🚨 FINAL FAILURE after {max_retries+1} attempts — saved to failed folder")

            return final_output

        # ❌ Failed but retries remain → prepare feedback for next loop iteration
        print(f"\n❌ Validation failed on attempt {attempt+1}, retrying with feedback...")
        old_output = json.dumps(final_output, indent=2)
        old_validation = res_val


W0901 19:11:52.319000 3514957 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0901 19:11:52.333000 3514957 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

## Test 1

In [3]:
import os

folder_path = r"/home/cipl/users/OCR_Project/Solution_15_04/Jeeva/confidence_logic/Met_8_pdf"

for filename in os.listdir(folder_path):
    if filename.lower().endswith(".pdf"):
        pdf_path = os.path.join(folder_path, filename)

        print(f"\n{'='*80}")
        print(f"Processing: {filename}")
        print(f"{'='*80}")

        try:
            run_pipeline(pdf_path)
            print(f"✅ Completed: {filename}")

        except Exception as e:
            print(f"❌ Failed: {filename}")
            print(f"Error: {e}")


Processing: Pmt_EOP_837707897.pdf
Processing PDF: Pmt_EOP_837707897.pdf  (ID: 837707897)
claim status : not denied

--------------------------------------------------
ATTEMPT 1 | Feedback: False
--------------------------------------------------
Building prompt!
Claim ID : 6022687984 99 018
Patient  : KEVIN CONLEY
------------------------------------------------------------
  Saved image: EOB_OUTPUT/Metlife/837707897/6022687984_99_018.png


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Both `max_new_tokens` (=1500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  Processing image 1: EOB_OUTPUT/Metlife/837707897/6022687984_99_018.png
  → Extracted: KEVIN CONLEY

  Running validation...

🔍 Validation for [KEVIN CONLEY]
---------------------------------------------------------------------------
✅ total_submitted           computed=367.0      | extracted=367.0      match
✅ total_allowed             computed=189.0      | extracted=189.0      match
✅ total_paid                computed=94.5       | extracted=94.5       match
✅ total_patient_owes        computed=94.5       | extracted=94.5       match
---------------------------------------------------------------------------
  ✅ [KEVIN CONLEY] Validation PASSED

✅ Success output + pdf + crops saved: EOB_output_success/Metlife/837707897

📁 Cropped images : EOB_OUTPUT/Metlife/837707897
✅ Success json   : EOB_output_success/Metlife/837707897/837707897_output.json
⚠  Failed json    : None

✅ SUCCESS on attempt 1
✅ Completed: Pmt_EOP_837707897.pdf

Processing: Pmt_EOP_838363708.pdf
Processing PDF: Pmt_EO

[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Both `max_new_tokens` (=1500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  Processing image 1: EOB_OUTPUT/Metlife/838363708/6022799976_99_009.png
  → Extracted: DENIS CALDERA

  Running validation...

🔍 Validation for [DENIS CALDERA]
---------------------------------------------------------------------------
✅ total_submitted           computed=167.0      | extracted=167.0      match
✅ total_allowed             computed=85.0       | extracted=85.0       match
✅ total_paid                computed=85.0       | extracted=85.0       match
✅ total_patient_owes        computed=0.0        | extracted=0.0        match
---------------------------------------------------------------------------
  ✅ [DENIS CALDERA] Validation PASSED

✅ Success output + pdf + crops saved: EOB_output_success/Metlife/838363708

📁 Cropped images : EOB_OUTPUT/Metlife/838363708
✅ Success json   : EOB_output_success/Metlife/838363708/838363708_output.json
⚠  Failed json    : None

✅ SUCCESS on attempt 1
✅ Completed: Pmt_EOP_838363708.pdf

Processing: Pmt_EOP_838363707.pdf
Processing PDF: Pmt

...

  Processing image 1: EOB_OUTPUT/Metlife/839405948/6030299984_99_017.png
  → Extracted: SHAUN FISHER

  Running validation...

🔍 Validation for [SHAUN FISHER]
---------------------------------------------------------------------------
✅ total_submitted           computed=298.0      | extracted=298.0      match
✅ total_allowed             computed=140.0      | extracted=140.0      match
✅ total_paid                computed=110.3      | extracted=110.3      match
✅ total_patient_owes        computed=29.7       | extracted=29.7       match
---------------------------------------------------------------------------
  ✅ [SHAUN FISHER] Validation PASSED

✅ Success output + pdf + crops saved: EOB_output_success/Metlife/839405948

📁 Cropped images : EOB_OUTPUT/Metlife/839405948
✅ Success json   : EOB_output_success/Metlife/839405948/839405948_output.json
⚠  Failed json    : None

✅ SUCCESS on attempt 1
✅ Completed: Pmt_EOP_839405948.pdf

Processing: Pmt_EOP_837701591.pdf
Processing PDF: Pmt_EO

[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Both `max_new_tokens` (=1500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  Processing image 1: EOB_OUTPUT/Metlife/837701591/6022699929_99_009.png
  → Extracted: MADAN MOHAN POLAM

  Running validation...

🔍 Validation for [MADAN MOHAN POLAM]
---------------------------------------------------------------------------
✅ total_submitted           computed=469.0      | extracted=469.0      match
✅ total_allowed             computed=243.0      | extracted=243.0      match
✅ total_paid                computed=243.0      | extracted=243.0      match
✅ total_patient_owes        computed=0.0        | extracted=0.0        match
---------------------------------------------------------------------------
  ✅ [MADAN MOHAN POLAM] Validation PASSED

✅ Success output + pdf + crops saved: EOB_output_success/Metlife/837701591

📁 Cropped images : EOB_OUTPUT/Metlife/837701591
✅ Success json   : EOB_output_success/Metlife/837701591/837701591_output.json
⚠  Failed json    : None

✅ SUCCESS on attempt 1
✅ Completed: Pmt_EOP_837701591.pdf

Processing: Pmt_EOP_836810655.pdf
Process

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Both `max_new_tokens` (=1500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  Processing image 1: EOB_OUTPUT/Metlife/836810655/6022430675_99_019.png
  → Extracted: ASHVATH VASANTHAKUMAR

  Running validation...

🔍 Validation for [ASHVATH VASANTHAKUMAR]
---------------------------------------------------------------------------
✅ total_submitted           computed=531.0      | extracted=531.0      match
✅ total_allowed             computed=274.0      | extracted=274.0      match
✅ total_paid                computed=274.0      | extracted=274.0      match
✅ total_patient_owes        computed=0.0        | extracted=0.0        match
---------------------------------------------------------------------------
  ✅ [ASHVATH VASANTHAKUMAR] Validation PASSED

✅ Success output + pdf + crops saved: EOB_output_success/Metlife/836810655

📁 Cropped images : EOB_OUTPUT/Metlife/836810655
✅ Success json   : EOB_output_success/Metlife/836810655/836810655_output.json
⚠  Failed json    : None

✅ SUCCESS on attempt 1
✅ Completed: Pmt_EOP_836810655.pdf


In [3]:
run_pipeline(r"/home/cipl/users/OCR_Project/Solution_15_04/Jeeva/Metlife/Met_pdf_100/Pmt_EOP_836819123.pdf")

Processing PDF: Pmt_EOP_836819123.pdf  (ID: 836819123)
claim status : not denied

--------------------------------------------------
ATTEMPT 1 | Feedback: False
--------------------------------------------------
Building prompt!
Claim ID : 6022599973 99 009
Patient  : ANTONIO SILVA
------------------------------------------------------------
  Saved image: EOB_OUTPUT/Metlife/836819123/6022599973_99_009.png


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Keyword argument `temperature` is not a valid argument for this processor and will be ignored.
[transformers] Keyword argument `do_sample` is not a valid argument for this processor and will be ignored.
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


  Processing image 1: EOB_OUTPUT/Metlife/836819123/6022599973_99_009.png


[transformers] Both `max_new_tokens` (=1500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  → Extracted: ANTONIO SILVA

  Running validation...

🔍 Validation for [ANTONIO SILVA]
---------------------------------------------------------------------------
✅ total_submitted           computed=1119.0     | extracted=1119.0     match
✅ total_allowed             computed=464.0      | extracted=464.0      match
✅ total_paid                computed=358.4      | extracted=358.4      match
✅ total_patient_owes        computed=204.6      | extracted=204.6      match
---------------------------------------------------------------------------
  ✅ [ANTONIO SILVA] Validation PASSED

✅ Success output + pdf + crops saved: EOB_output_success/Metlife/836819123

📁 Cropped images : EOB_OUTPUT/Metlife/836819123
✅ Success json   : EOB_output_success/Metlife/836819123/836819123_output.json
⚠  Failed json    : None

✅ SUCCESS on attempt 1


[{'eob_id': '836819123',
  'payor': 'Metlife',
  'claim_status': 'not denied',
  'confidence_score': 84.0,
  'pdf_file': 'Pmt_EOP_836819123.pdf',
  'patients': [{'patient_name': 'ANTONIO SILVA',
    'relationship': 'Self',
    'dob': '',
    'date_of_service': '02/24/26',
    'services': [{'service_code': 'D0120',
      'submitted_amount': '89.00',
      'allowed_amount': '41.00',
      'paid_amount': '41.00',
      'deductible': '',
      'patient_owes': '0.00'},
     {'service_code': 'D0220',
      'submitted_amount': '45.00',
      'allowed_amount': '24.00',
      'paid_amount': '24.00',
      'deductible': '',
      'patient_owes': '0.00'},
     {'service_code': 'D0230',
      'submitted_amount': '37.00',
      'allowed_amount': '20.00',
      'paid_amount': '20.00',
      'deductible': '',
      'patient_owes': '0.00'},
     {'service_code': 'D0274',
      'submitted_amount': '99.00',
      'allowed_amount': '51.00',
      'paid_amount': '51.00',
      'deductible': '',
      'pat